# CUDA C++ Course — Google Colab edition

This notebook walks through **23 progressive CUDA exercises** from the [course repository](https://github.com/lsawicki-cdv/course-accelerating-apps-nvidia-cuda), compiled and run on a free Google Colab GPU.

## Before you start

1. Open **Runtime → Change runtime type** and pick **GPU** as the hardware accelerator.
2. The default GPU on the free tier is a **Tesla T4** (compute capability 7.5). That is enough for exercises **1–17 and 21–23**.
3. Exercises **18–20** use Tensor Cores and require compute capability **≥ 8.9** (Ada Lovelace). To run those, switch to an **L4** or **A100** runtime (Colab Pro).
4. Run the cells in order from top to bottom. The setup section below clones the repo; every later cell compiles a single `.cu` file from it.

## How the notebook works

- Each exercise gets a short explanation, then one cell per `.cu` file that compiles and runs it with `nvcc`.
- Compiled binaries land next to the source (`examples/<dir>/out.bin`).
- The profiling section at the end uses `nsys profile` to inspect kernel timing on a few key exercises.

## 1. Environment setup

### 1.1 Check the GPU

In [ ]:
!nvidia-smi

### 1.2 Check the CUDA compiler

In [ ]:
!nvcc --version

### 1.3 Detect compute capability

We read the GPU's compute capability and set two Python flags that later cells use to gate tensor-core exercises.

In [ ]:
import subprocess

out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"]).decode().strip()
GPU_NAME, cc_str = [x.strip() for x in out.split(",")]
COMPUTE_CAP = float(cc_str)
HAS_TENSOR_CORES = COMPUTE_CAP >= 8.0
HAS_SM89 = COMPUTE_CAP >= 8.9

print(f"GPU:              {GPU_NAME}")
print(f"Compute cap:      {COMPUTE_CAP}")
print(f"Tensor cores:     {HAS_TENSOR_CORES}")
print(f"Ada (sm_89) path: {HAS_SM89}  (needed for exercises 18-20)")

### 1.4 Clone the course repo

The notebook compiles `.cu` files directly from the cloned tree — it does not embed the source.

In [ ]:
import os
if not os.path.isdir('course-accelerating-apps-nvidia-cuda'):
    !git clone https://github.com/lsawicki-cdv/course-accelerating-apps-nvidia-cuda.git
%cd course-accelerating-apps-nvidia-cuda
!ls examples | head

## 2. Kernel basics (exercises 1–6)

### Exercise 1 — GPU Hello World

Launches a minimal `__global__` kernel. Shows the three pieces every CUDA program has: kernel definition, triple-chevron launch `<<<blocks, threads>>>`, and `cudaDeviceSynchronize()`.

In [ ]:
!cd examples/1-gpu-hello-world && nvcc -o out.bin hello-world-gpu.cu -run

### Exercise 2 — Thread and block indexing

Introduces `threadIdx.x` and `blockIdx.x`, and shows how to use a conditional so only one thread prints.

In [ ]:
!cd examples/2-cuda-kernel-idx && nvcc -o out.bin thread-and-block-idx.cu -run

### Exercise 3 — Loops in a kernel

Two variants: a single-block loop and a multi-block loop, to contrast how work is distributed as you add blocks.

In [ ]:
!cd examples/3-loops && nvcc -o out.bin single-block-loop-gpu.cu -run

In [ ]:
!cd examples/3-loops && nvcc -o out.bin multiple-block-loop-gpu.cu -run

### Exercise 4 — Memory allocation and block configuration

`cudaMallocManaged` for Unified Memory; picking a block size and computing the grid size for arbitrary N.

In [ ]:
!cd examples/4-allocation && nvcc -o out.bin double-elements-gpu.cu -run

In [ ]:
!cd examples/4-allocation && nvcc -o out.bin block-config.cu -run

### Exercise 5 — Grid-stride loop

The canonical pattern for letting one thread process many elements: `for (i = tid; i < N; i += stride)`.

In [ ]:
!cd examples/5-grid-stride && nvcc -o out.bin grid-stride-double.cu -run

### Exercise 6 — Error handling

How to check `cudaGetLastError()` after a kernel launch and wrap calls in a `checkCuda()` helper. Silent GPU errors are the #1 reason kernels seem to do nothing.

In [ ]:
!cd examples/6-errors && nvcc -o out.bin error-handling.cu -run

## 3. Vector and matrix operations (exercises 7–8)

### Exercise 7 — Vector addition: CPU vs GPU

Classic first parallel benchmark. Run both to compare wall-clock time.

In [ ]:
!cd examples/7-vector && nvcc -o out.bin vector-add-cpu.cu -run

In [ ]:
!cd examples/7-vector && nvcc -o out.bin vector-add-gpu.cu -run

### Exercise 8 — 2D matrix multiply (1024×1024)

Naive O(N³) matmul with a 2D grid of 2D blocks. CPU reference plus GPU version.

In [ ]:
!cd examples/8-matrix-multiply && nvcc -o out.bin matrix-multiply-2d-cpu.cu -run

In [ ]:
!cd examples/8-matrix-multiply && nvcc -o out.bin matrix-multiply-2d-gpu.cu -run

## 4. Unified Memory and prefetching (exercises 9–10)

### Exercise 9 — Prefetch vs no-prefetch

Same vector-add, two versions: with and without `cudaMemPrefetchAsync`. Shows the cost of on-demand page migration.

In [ ]:
!cd examples/09-vector-add-prefetch && nvcc -o out.bin vector-add-no-prefetch.cu -run

In [ ]:
!cd examples/09-vector-add-prefetch && nvcc -o out.bin vector-add-prefetch.cu -run

### Exercise 10 — Initialize on CPU vs GPU

Compares initializing Unified Memory on the host versus with a kernel. The GPU-init version avoids the first-touch migration penalty.

In [ ]:
!cd examples/10-init-kernel && nvcc -o out.bin init-kernel-cpu.cu -run

In [ ]:
!cd examples/10-init-kernel && nvcc -o out.bin init-kernel-gpu.cu -run

## 5. CUDA streams (exercises 11–13)

### Exercise 11 — Sync vs async print

Two kernels, default stream (serialized) vs explicit streams (concurrent). Output ordering is the clearest way to see streams in action.

In [ ]:
!cd examples/11-stream-intro && nvcc -o out.bin print-numbers-sync.cu -run

In [ ]:
!cd examples/11-stream-intro && nvcc -o out.bin print-numbers-async.cu -run

### Exercise 12 — Stream-based initialization

Three versions: no streams, per-section streams, and a sliced-workload pattern that overlaps compute with the next slice's setup.

In [ ]:
!cd examples/12-stream-init && nvcc -o out.bin no-stream-init.cu -run

In [ ]:
!cd examples/12-stream-init && nvcc -o out.bin stream-init.cu -run

In [ ]:
!cd examples/12-stream-init && nvcc -o out.bin stream-sliced.cu -run

### Exercise 13 — SM-aware block configuration

Query the device for `multiProcessorCount` and size the grid so every SM gets work — foundational when mixing streams.

In [ ]:
!cd examples/13-vector-add-sm-blocks && nvcc -o out.bin vector-add-SM-blocks.cu -run

## 6. Advanced memory (exercises 14–17)

### Exercise 14 — Page faults in Unified Memory

Instrument migrations to see when the driver page-faults host→device and device→host.

In [ ]:
!cd examples/14-unified-memory-page-faults && nvcc -o out.bin page-faults.cu -run

### Exercise 15 — Prefetch strategy

Explicitly stage data to the device before the kernel runs to eliminate page-fault overhead.

In [ ]:
!cd examples/15-unified-memory-prefetch && nvcc -o out.bin vector-add-prefetch.cu -run

### Exercise 16 — Memory coalescing

Row-major vs column-major access on a 1024×1024 matrix. The non-coalesced version runs an order of magnitude slower — this is one of the biggest single wins in CUDA programming.

In [ ]:
!cd examples/16-memory-coalescing && nvcc -o out.bin memory-coalescing.cu -run

### Exercise 17 — Tiled matrix multiply with shared memory

Uses `__shared__` tiles and `__syncthreads()` to reduce global-memory loads by a factor of `TILE_SIZE`. Compare the timing against exercise 8's naive version.

In [ ]:
!cd examples/17-tiled-matrix-multiply && nvcc -o out.bin tiled-matrix-multiply.cu -run

## 7. Tensor cores (exercises 18–20)

> ⚠️ **These exercises require compute capability ≥ 8.9 (Ada/Hopper).** The free-tier Colab T4 will not work. Each cell is guarded by `HAS_SM89` and will skip cleanly if your GPU is too old.

### Exercise 18 — Tensor core intro (WMMA API)

Warp-level 16×16 FP16 → FP32 matrix multiply-accumulate using `nvcuda::wmma`.

In [ ]:
if HAS_SM89:
    !cd examples/18-tensor-cores-intro && nvcc -arch=sm_89 -o out.bin tensor-cores-intro.cu -run
else:
    print("Skipped: this exercise needs compute capability >= 8.9 (Ada/Hopper).")
    print("In Colab: Runtime -> Change runtime type -> L4 or A100 GPU, then re-run from the top.")

### Exercise 19 — Tensor cores + shared memory

Stage 32×32 tiles in shared memory, then feed them to WMMA fragments. Combines the shared-memory pattern from exercise 17 with tensor-core MMA.

In [ ]:
if HAS_SM89:
    !cd examples/19-tensor-cores-shared-memory && nvcc -arch=sm_89 -o out.bin tensor-cores-shared-memory.cu -run
else:
    print("Skipped: this exercise needs compute capability >= 8.9 (Ada/Hopper).")
    print("In Colab: Runtime -> Change runtime type -> L4 or A100 GPU, then re-run from the top.")

### Exercise 20 — SGEMM optimization progression (+ cuBLAS)

Six progressively faster SGEMM kernels benchmarked against cuBLAS. Needs `-lcublas`.

In [ ]:
if HAS_SM89:
    !cd examples/20-sgemm-optimizations && nvcc -arch=sm_89 -lcublas -o out.bin sgemm-optimizations.cu -run
else:
    print("Skipped: this exercise needs compute capability >= 8.9 (Ada/Hopper).")
    print("In Colab: Runtime -> Change runtime type -> L4 or A100 GPU, then re-run from the top.")

## 8. Image convolution (exercises 21–23)

### Exercise 21 — Naive 2D convolution

Apply a 3×3 filter to a synthetic 2048×2048 grayscale image. The kernel itself is fast (~0.2 ms); transfer time dwarfs it.

In [ ]:
!cd examples/21-image-convolution-naive && nvcc -o out.bin image-convolution-naive.cu -run

### Exercise 22 — Constant memory + pinned host memory

The filter coefficients go in `__constant__` memory; the host buffers use `cudaHostAlloc` for faster DMA. Watch H2D/D2H times shrink.

In [ ]:
!cd examples/22-image-convolution-memory && nvcc -o out.bin image-convolution-memory.cu -run

### Exercise 23 — Shared-memory halo tiling

Cooperative load of a tile plus its halo into shared memory so each output pixel hits shared — not global — memory for its neighbors. For small filters (3×3) the overhead doesn't pay off; for 15×15 you see roughly a 2× speedup.

In [ ]:
!cd examples/23-image-convolution-shared-memory && nvcc -o out.bin image-convolution-shared-memory.cu -run

## 9. Profiling with Nsight Systems

`nsys profile --stats=true` runs the binary and prints a timeline summary: kernel time, memcpy time, API overhead. It's the fastest way to see where your program actually spends time.

The reports below also write `.nsys-rep` files you can download and open in the Nsight Systems desktop app for the full interactive timeline.

### 9.1 Exercise 16 — cost of non-coalesced access

Look at the kernel time column for the two kernels — the row-wise (coalesced) kernel should be dramatically faster.

In [ ]:
!cd examples/16-memory-coalescing && nvcc -o out.bin memory-coalescing.cu && nsys profile --stats=true --force-overwrite=true -o report_coalescing ./out.bin

### 9.2 Exercise 17 — shared-memory tiling payoff

Kernel time for the tiled matmul versus the naive version from exercise 8. The reduction in global-memory loads is what you're paying for with the `__shared__` buffer.

In [ ]:
!cd examples/17-tiled-matrix-multiply && nvcc -o out.bin tiled-matrix-multiply.cu && nsys profile --stats=true --force-overwrite=true -o report_tiled ./out.bin

### 9.3 Exercise 22 — transfers dominate kernel time

This is the most important lesson in the course: even with an optimized kernel, `cudaMemcpy` H2D + D2H can be 10× the compute time. Optimization effort should follow where the clock actually goes.

In [ ]:
!cd examples/22-image-convolution-memory && nvcc -o out.bin image-convolution-memory.cu && nsys profile --stats=true --force-overwrite=true -o report_conv_mem ./out.bin

## Next steps

- Modify any `.cu` file in the file browser on the left and re-run its cell to see your change.
- Open the generated `.nsys-rep` files locally in Nsight Systems for the full timeline view.
- Browse the full course, including the `cuda-webcam-filter` production template, at [github.com/lsawicki-cdv/course-accelerating-apps-nvidia-cuda](https://github.com/lsawicki-cdv/course-accelerating-apps-nvidia-cuda).